##### Parameter Cell

In [5]:
batch_id = ""

StatementMeta(, 90b4bf4f-ef7c-4944-8b3f-0063a2f378cc, 7, Finished, Available, Finished, False)

##### Setup

In [6]:
import sys
sys.path.insert(0, "/lakehouse/default/Files/code")

from bikeshare import control
from bikeshare.audit import RunLogger
from bikeshare.config import make_config

spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
cfg = make_config(batch_id=batch_id or None)
control.ensure_control_plane(spark)
log = RunLogger(spark, cfg)

StatementMeta(, 90b4bf4f-ef7c-4944-8b3f-0063a2f378cc, 8, Finished, Available, Finished, False)

##### Run

In [7]:
from bikeshare import gold
failed = gold.run(spark, cfg, log)
if failed:
    raise Exception(f"Critical data quality checks failed: {failed}")

StatementMeta(, 90b4bf4f-ef7c-4944-8b3f-0063a2f378cc, 9, Finished, Available, Finished, False)

  [gold   ] dim_date                     SUCCESS  read=0       written=0       rejected=0       2.4s
  [gold   ] dim_rider_type + dim_bike_type SUCCESS  read=0       written=4       rejected=0       6.5s
  [gold   ] dim_station (SCD2)           SUCCESS  read=2       written=2       rejected=0      22.3s
  [gold   ] dim_station_current          SUCCESS  read=0       written=40      rejected=0       5.2s
  [gold   ] dim_weather_hour             SUCCESS  read=0       written=504     rejected=0       5.0s
  [gold   ] dim_event                    SUCCESS  read=0       written=21      rejected=0       4.3s
  [gold   ] fact_trip                    SUCCESS  read=15221   written=15221   rejected=0      19.7s
  [gold   ] mart_daily_kpis              SUCCESS  read=0       written=16      rejected=0       9.3s
  [gold   ] mart_station_day             SUCCESS  read=0       written=639     rejected=0       7.0s
  [gold   ] mart_hourly_demand           SUCCESS  read=0       written=375     rejected=0

##### Checks

In [8]:
for t in ["dim_date","dim_station","dim_station_current","dim_weather_hour","dim_event","dim_rider_type","dim_bike_type",
          "fact_trip","mart_daily_kpis","mart_station_day","mart_hourly_demand"]:
    print(t, spark.table(f"gold.{t}").count())

spark.sql("""SELECT rule_name, rows_failed, status FROM audit.dq_results
             WHERE layer='gold' ORDER BY checked_at DESC, rule_name""").show(truncate=False)
spark.table("gold.fact_trip").show(5, truncate=False)

StatementMeta(, 90b4bf4f-ef7c-4944-8b3f-0063a2f378cc, 10, Finished, Available, Finished, False)

dim_date 730
dim_station 43
dim_station_current 40
dim_weather_hour 504
dim_event 21
dim_rider_type 2
dim_bike_type 2
fact_trip 18851
mart_daily_kpis 21
mart_station_day 839
mart_hourly_demand 491
+--------------------------+-----------+------+
|rule_name                 |rows_failed|status|
+--------------------------+-----------+------+
|no_overlap_or_gap         |0          |PASS  |
|one_current_row_per_key   |0          |PASS  |
|end_station_resolved      |70         |WARN  |
|start_station_resolved    |17         |WARN  |
|weather_key_in_dim_weather|0          |PASS  |
|date_key_in_dim_date      |0          |PASS  |
|trip_id_unique            |0          |PASS  |
|revenue_matches_silver    |0          |PASS  |
|rowcount_matches_silver   |0          |PASS  |
|no_overlap_or_gap         |0          |PASS  |
|one_current_row_per_key   |0          |PASS  |
|end_station_resolved      |8          |WARN  |
|start_station_resolved    |2          |WARN  |
|weather_key_in_dim_weather|0      

##### History Check

In [9]:
spark.sql("""SELECT station_id, station_name, capacity, station_status, _valid_from, _valid_to, _is_current
             FROM gold.dim_station
             WHERE station_id IN ('S001','S004') ORDER BY station_id, _valid_from""").show(truncate=False)
spark.table("gold.dim_station").count()     # expect 43: 41 + 2 new versions

StatementMeta(, 90b4bf4f-ef7c-4944-8b3f-0063a2f378cc, 11, Finished, Available, Finished, False)

+----------+-----------------------+--------+--------------+-------------------+-------------------+-----------+
|station_id|station_name           |capacity|station_status|_valid_from        |_valid_to          |_is_current|
+----------+-----------------------+--------+--------------+-------------------+-------------------+-----------+
|S001      |Flinders Street Station|30      |active        |2025-03-01 00:00:00|2026-08-13 03:00:00|false      |
|S001      |Flinders Street Station|40      |active        |2026-08-13 03:00:00|9999-12-31 00:00:00|true       |
|S004      |Queen Victoria Market  |24      |active        |2025-03-04 00:00:00|2026-08-21 03:00:00|false      |
|S004      |Queen Victoria Market  |24      |maintenance   |2026-08-21 03:00:00|9999-12-31 00:00:00|true       |
+----------+-----------------------+--------+--------------+-------------------+-------------------+-----------+



43